<a href="https://colab.research.google.com/github/Saibhossain/face-generation-model/blob/main/Text_to_Face_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install diffusers
!pip install transformers accelerate scipy safetensors


# explore the model (runwayml/stable-diffusion-v1-5)

## function

In [5]:
import torch
from diffusers import StableDiffusionPipeline
from huggingface_hub import model_info
import psutil
import os

# --- CONFIGURATION ---
MODEL_ID = "runwayml/stable-diffusion-v1-5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def print_section(title):
    print(f"\n{'='*20} {title.upper()} {'='*20}")

def get_hub_metadata():
    """Fetches metadata from the Hugging Face Hub API"""
    print_section("1. Hugging Face Hub Metadata")
    try:
        info = model_info(MODEL_ID)
        print(f"Model ID:       {info.modelId}")
        print(f"Author:         {info.author}")
        print(f"Downloads:      {info.downloads:,}")
        print(f"Likes:          {info.likes:,}")
        print(f"Library:        {info.library_name}")

        # Tags often contain license and task info
        print(f"Tags:           {', '.join(info.tags[:5])}...")

    except Exception as e:
        print(f"Error fetching Hub metadata: {e}")

def count_params(model):
    """Counts trainable parameters in a PyTorch model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def inspect_pipeline_components():
    """Loads the model and dissects its internals"""
    print_section("2. Architecture & Components")
    print(f"Loading model into {DEVICE} (float16 for efficiency)...")

    # Load pipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    # --- A. TEXT ENCODER (CLIP) ---
    print("\n--- [Component 1: Text Encoder (CLIP)] ---")
    print("This converts your prompt into vectors.")
    text_config = pipe.text_encoder.config
    print(f"   • Model Type:      {text_config.model_type}")
    print(f"   • Vocab Size:      {text_config.vocab_size:,}")
    print(f"   • Hidden Size:     {text_config.hidden_size}")
    print(f"   • Layers:          {text_config.num_hidden_layers}")
    print(f"   • Parameters:      {count_params(pipe.text_encoder):,}")

    # --- B. UNET (The Noise Predictor) ---
    print("\n--- [Component 2: UNet (The Engine)] ---")
    print("This predicts noise to subtract from the image.")
    unet_config = pipe.unet.config
    print(f"   • Sample Size:     {unet_config.sample_size} (Internal processing resolution)")
    print(f"   • In Channels:     {unet_config.in_channels} (Latent channels)")
    print(f"   • Cross Attn Dim:  {unet_config.cross_attention_dim} (Text embedding size)")
    print(f"   • Attn Head Dim:   {unet_config.attention_head_dim}")
    print(f"   • Parameters:      {count_params(pipe.unet):,}")

    # --- C. VAE (Variational Autoencoder) ---
    print("\n--- [Component 3: VAE (The Compressor)] ---")
    print("This compresses images to latents and decompresses them back.")
    vae_config = pipe.vae.config
    print(f"   • Latent Channels: {vae_config.latent_channels}")
    print(f"   • Block Out Ch:    {vae_config.block_out_channels}")
    print(f"   • Parameters:      {count_params(pipe.vae):,}")

    # --- D. SCHEDULER ---
    print("\n--- [Component 4: Scheduler] ---")
    print(f"   • Default Type:    {pipe.scheduler.__class__.__name__}")
    print(f"   • Timesteps:       {pipe.scheduler.config.num_train_timesteps} (Training steps)")
    print(f"   • Beta Schedule:   {pipe.scheduler.config.beta_schedule}")

    return pipe

def qualitative_test(pipe):
    """Runs a generation test"""
    print_section("3. Qualitative Evaluation (Inference Test)")

    prompt = "A high-tech robot painting a canvas, detailed, 8k, cyberpunk style"
    print(f"Generating image for prompt: '{prompt}'")

    image = pipe(prompt, num_inference_steps=25).images[0]

    save_path = "sd15_evaluation_sample.png"
    image.save(save_path)
    print(f"Test image saved to: {save_path}")
    print("Check this image to evaluate visual quality.")

## METADATA

In [6]:
get_hub_metadata()


==================== 1. HUGGING FACE HUB METADATA ====================
Model ID:       stable-diffusion-v1-5/stable-diffusion-v1-5
Author:         stable-diffusion-v1-5
Downloads:      2,177,579
Likes:          931
Library:        diffusers
Tags:           diffusers, safetensors, stable-diffusion, stable-diffusion-diffusers, text-to-image...


## ARCHITECTURE

In [7]:
pipeline_obj = inspect_pipeline_components()


==================== 2. ARCHITECTURE & COMPONENTS ====================
Loading model into cuda (float16 for efficiency)...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]


--- [Component 1: Text Encoder (CLIP)] ---
This converts your prompt into vectors.
   • Model Type:      clip_text_model
   • Vocab Size:      49,408
   • Hidden Size:     768
   • Layers:          12
   • Parameters:      123,060,480

--- [Component 2: UNet (The Engine)] ---
This predicts noise to subtract from the image.
   • Sample Size:     64 (Internal processing resolution)
   • In Channels:     4 (Latent channels)
   • Cross Attn Dim:  768 (Text embedding size)
   • Attn Head Dim:   8
   • Parameters:      859,520,964

--- [Component 3: VAE (The Compressor)] ---
This compresses images to latents and decompresses them back.
   • Latent Channels: 4
   • Block Out Ch:    [128, 256, 512, 512]
   • Parameters:      83,653,863

--- [Component 4: Scheduler] ---
   • Default Type:    PNDMScheduler
   • Timesteps:       1000 (Training steps)
   • Beta Schedule:   scaled_linear


## TEST

In [8]:
qualitative_test(pipeline_obj)


==================== 3. QUALITATIVE EVALUATION (INFERENCE TEST) ====================
Generating image for prompt: 'A high-tech robot painting a canvas, detailed, 8k, cyberpunk style'


  0%|          | 0/25 [00:00<?, ?it/s]

Test image saved to: sd15_evaluation_sample.png
Check this image to evaluate visual quality.


# my code

In [1]:
import torch
from IPython.display import display
import sys
import subprocess
from diffusers import StableDiffusionPipeline

# --- 2. LOAD MODEL ---
# We use Stable Diffusion v1.5 because it is excellent at portraits and human faces
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Model: {model_id}...")
# float16 makes it faster and use less memory on Colab GPUs
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to(device)

# Optimization for faster generation
pipe.enable_attention_slicing()

# --- 3. GENERATION FUNCTION ---
def generate_face(prompt, negative_prompt="cartoon, 3d, disfigured, bad art"):
    print(f"\n✨ Generating: '{prompt}'...")

    # The pipeline handles all the math (denoising, latents, decoding) automatically
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        height=512,
        width=512,
        num_inference_steps=30 # 30 is a sweet spot for speed/quality
    ).images[0]

    display(image)
    return image

# --- 4. USER INPUT ---
# Type your description here!
user_prompt = "A cinematic portrait of a futuristic cyberpunk warrior with neon blue eyes, 8k resolution, photorealistic"


# Run Generation
img = generate_face(user_prompt)

🚀 Loading Model: runwayml/stable-diffusion-v1-5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!



✨ Generating: 'None'...


ValueError: Provide either `prompt` or `prompt_embeds`. Cannot leave both `prompt` and `prompt_embeds` undefined.